In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [2]:
def reorder(ad1, ad2):
    shared_barcodes = ad1.obs_names.intersection(ad2.obs_names)
    ad1 = ad1[shared_barcodes].copy()
    ad2 = ad2[shared_barcodes].copy()
    return ad1, ad2

def load_peak_expr(_dir):
    data = sio.mmread(join(_dir, 'data.mtx'))
    cname = pd.read_csv(join(_dir, 'barcode.csv'), index_col=0)['x'].to_list()
    feat = pd.read_csv(join(_dir, 'feat.csv'), index_col=0)['x'].to_list()
    ad = sc.AnnData(sps.csr_matrix(data.T))
    ad.obs_names = cname
    ad.var_names = feat
    return ad

In [3]:
data_dir = '../../../data/raw/MB-5M/mouse_brain_rna+atac'

df1_rna = pd.read_csv(join(data_dir, 'rna+atac/GSM6204636_MouseBrain_20um_matrix.tsv'), sep='\t')
df1_spatial_pos = pd.read_csv(join(data_dir, 'rna+atac/GSM6204623_MouseBrain_20um_spatial_rna_part/tissue_positions_list.csv'), header=None, index_col=0)
ad1_rna = sc.AnnData(df1_rna.T, obsm={'spatial': df1_spatial_pos.loc[df1_rna.columns, [2, 3]].values})

ad1_atac = load_peak_expr(join(data_dir, 'rna+atac/For_Imputation_Task/GSM6204623_peak_data'))
df1_atac_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6204623_MouseBrain_20um_spatial_rna_part/tissue_positions_list.csv'), index_col=0, header=None)
ad1_atac.obsm['spatial'] = df1_atac_spatial.loc[ad1_atac.obs_names, [2, 3]].values
ad1_rna, ad1_atac = reorder(ad1_rna, ad1_atac)

# ===
df2_rna = pd.read_csv(join(data_dir, 'rna+atac/GSM6753041_MouseBrain_20um_repATAC_matrix.tsv'), sep='\t')
df2_rna_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6753041_MouseBrain_20um_repATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad2_rna = sc.AnnData(df2_rna.T, obsm={'spatial': df2_rna_spatial.loc[df2_rna.columns, [2, 3]].values})

ad2_atac = load_peak_expr(join(data_dir, 'rna+atac/For_Imputation_Task/GSM6758284_peak_data'))
df2_atac_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6753041_MouseBrain_20um_repATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad2_atac.obsm['spatial'] = df2_atac_spatial.loc[ad2_atac.obs_names, [2, 3]].values
ad2_rna, ad2_atac = reorder(ad2_rna, ad2_atac)

# ===
df3_rna = pd.read_csv(join(data_dir, 'rna+atac/GSM6753043_MouseBrain_20um_100barcodes_ATAC_matrix.tsv'), sep='\t')
df3_rna_spatial = pd.read_csv(join(data_dir, 'rna+atac/GSM6753043_MouseBrain_20um_100barcodes_ATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad3_rna = sc.AnnData(df3_rna.T, obsm={'spatial': df3_rna_spatial.loc[df3_rna.columns, [2, 3]].values})

ad3_atac = load_peak_expr(join(data_dir, 'rna+atac/For_Imputation_Task/GSM6758285_peak_data'))
df3_atac_spatial = pd.read_csv(join(data_dir, 'rna+atac//GSM6753043_MouseBrain_20um_100barcodes_ATAC_spatial/tissue_positions_list.csv'), index_col=0, header=None)
ad3_atac.obsm['spatial'] = df3_atac_spatial.loc[ad3_atac.obs_names, [2, 3]].values
ad3_rna, ad3_atac = reorder(ad3_rna, ad3_atac)

shared_gene = ad1_rna.var_names.intersection(ad2_rna.var_names).intersection(ad3_rna.var_names)
shared_peak = ad1_atac.var_names.intersection(ad2_atac.var_names).intersection(ad3_atac.var_names)
ad1_rna = ad1_rna[:, shared_gene].copy(); ad2_rna = ad2_rna[:, shared_gene].copy(); ad3_rna = ad3_rna[:, shared_gene].copy()
ad1_atac = ad1_atac[:, shared_peak].copy(); ad2_atac = ad2_atac[:, shared_peak].copy(); ad3_atac = ad3_atac[:, shared_peak].copy()

ad1_rna.obs_names = [f's1-{_}' for _ in ad1_rna.obs_names]
ad1_atac.obs_names = [f's1-{_}' for _ in ad1_atac.obs_names]
ad2_rna.obs_names = [f's2-{_}' for _ in ad2_rna.obs_names]
ad2_atac.obs_names = [f's2-{_}' for _ in ad2_atac.obs_names]
ad3_rna.obs_names = [f's3-{_}' for _ in ad3_rna.obs_names]
ad3_atac.obs_names = [f's3-{_}' for _ in ad3_atac.obs_names]

ad1_rna.obs['src'] = ['s1']*ad1_rna.n_obs
ad1_atac.obs['src'] = ['s1']*ad1_atac.n_obs
ad2_rna.obs['src'] = ['s2']*ad2_rna.n_obs
ad2_atac.obs['src'] = ['s2']*ad2_atac.n_obs
ad3_rna.obs['src'] = ['s3']*ad3_rna.n_obs
ad3_atac.obs['src'] = ['s3']*ad3_atac.n_obs

In [4]:
ad_rna_all = sc.concat([ad1_rna, ad2_rna, ad3_rna])
ad_atac_all = sc.concat([ad1_atac, ad2_atac, ad3_atac])

sc.pp.highly_variable_genes(ad_rna_all, flavor='seurat_v3', n_top_genes=5000, batch_key='src')
hvg_names = ad_rna_all.var.query('highly_variable').index.to_numpy()

# ac.pp.tfidf(ad_atac_all, scale_factor=1e4)
sc.pp.highly_variable_genes(ad_atac_all, flavor='seurat_v3', n_top_genes=50000, batch_key='src')
hvp_names = ad_atac_all.var.query('highly_variable').index.to_numpy()

In [5]:
ad1_rna = ad1_rna[:, hvg_names].copy(); ad1_atac = ad1_atac[:, hvp_names].copy()
ad2_rna = ad2_rna[:, hvg_names].copy(); ad2_atac = ad2_atac[:, hvp_names].copy()
ad3_rna = ad3_rna[:, hvg_names].copy(); ad3_atac = ad3_atac[:, hvp_names].copy()

## filter feat names
filtered_atac_feats = [_ for _ in ad1_atac.var_names if _.startswith('chr')]
ad1_atac = ad1_atac[:, filtered_atac_feats].copy()
ad2_atac = ad2_atac[:, filtered_atac_feats].copy()
ad3_atac = ad3_atac[:, filtered_atac_feats].copy()

In [6]:
RNA_ADS = [ad1_rna, ad2_rna, ad3_rna]
ATAC_ADS = [ad1_atac, ad2_atac, ad3_atac]
mod_dict = {'rna': RNA_ADS, 'atac':ATAC_ADS}
n_batches = 3
mod_sets = ['rna', 'atac']

In [7]:
res = []
for i in range(n_batches):  # train test split
    print(f'cv={i+1}, missing=atac')
    mod_BatchDict = {'rna': RNA_ADS, 
                     'atac': [ATAC_ADS[bi] if bi!=i else None for bi in range(n_batches)]}
    input_key = 'dimred_bc'
    batch_key = 'src'
        
    Epigenome_preprocess(mod_BatchDict['atac'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
    RNA_preprocess(mod_BatchDict['rna'], favor='scanpy', batch_corr=True, n_hvg=5000, batch_key=batch_key, key=input_key)
    
    model = SpaMosaic(
        modBatch_dict=mod_BatchDict, input_key=input_key,
        batch_key=batch_key, 
        intra_knns=10, inter_knn_base=10, 
        w_g=0.8,
        seed=1234, 
        device='cuda:0'
    )
    
    model.train(net='wlgcn', lr=0.01, T=0.01, n_epochs=100)
    _ = model.infer_emb(mod_BatchDict, emb_key='emb', final_latent_key='merged_emb')

    for k in mod_BatchDict.keys():
        for ad in mod_BatchDict[k]:
            if ad is not None:
                ad.layers['counts'] = sps.csr_matrix(ad.X)

    # for knn in [10, 20, 30]:
    imp_dict = model.impute(mod_BatchDict, emb_key='emb', layer_key='counts', imp_knn=10)

    print(f'==> cv {i}, ATAC imputation: ')
    pr_X = imp_dict['atac'][i] 
    ad_pred = sc.AnnData(pr_X, obs=mod_dict['atac'][i].obs.copy(), var=mod_dict['atac'][i].var.copy())

    out_dir = "../../../results/imputations/SpaMosaic/MB_imputation/" #path to results
    os.makedirs(out_dir, exist_ok=True)
    ad_pred.write_h5ad(f'{out_dir}/cv{i}_imputedATAC.h5ad')

cv=1, missing=atac
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
Reach convergence after 4 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
batch0: ['rna']
batch1: ['rna', 'atac']
batch2: ['rna', 'atac']
------Calculating spatial graph...
The graph contains 23720 edges, 2372 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 24970 edges, 2497 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 92150 edges, 9215 cells.
10.0000 neighbors per cel

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:05<00:00, 17.84it/s]


impute atac-counts for batch-1
==> cv 0, ATAC imputation: 
cv=2, missing=atac
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
Reach convergence after 4 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
batch0: ['rna', 'atac']
batch1: ['rna']
batch2: ['rna', 'atac']
------Calculating spatial graph...
The graph contains 23720 edges, 2372 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 24970 edges, 2497 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph 

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:05<00:00, 18.12it/s]


impute atac-counts for batch-2
==> cv 1, ATAC imputation: 
cv=3, missing=atac
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
Reach convergence after 4 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
batch0: ['rna', 'atac']
batch1: ['rna', 'atac']
batch2: ['rna']
------Calculating spatial graph...
The graph contains 23720 edges, 2372 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 24970 edges, 2497 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph 

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 27.09it/s]


impute atac-counts for batch-3
==> cv 2, ATAC imputation: 
